# Sentinel-1 Preprocessing and LiDAR Collocation

This notebook processes extracted Sentinel-1 SAFE products and cuts windows corresponding to existing LiDAR patches. It supports VV and VH polarizations, converts calibrated values to dB for model input, and writes `s1_patch_<id>/t<i>.tif` directories.

The raw SAFE route below performs calibration and GCP-based warping, but it is **not full radiometric terrain correction**. For production work, use externally terrain-corrected GeoTIFFs from SNAP, ASF HyP3, or an equivalent processor.

In [ ]:
from pathlib import Path
import glob
import json
import xml.etree.ElementTree as ET
import numpy as np
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.warp import transform_bounds, calculate_default_transform, reproject, Resampling
from scipy.interpolate import RectBivariateSpline
import matplotlib.pyplot as plt

In [ ]:
REPO_DIR = Path('/Users/jessica/Desktop/project/Michel/RoughNet')
REGION = 'tuk'
LIDAR_DIR = REPO_DIR / 'input_data' / f'lidar_patches_{REGION}'
SAFE_ROOT = REPO_DIR / 'raw_data' / f'{REGION}_sentinel1_downloads'
CORRECTED_DIR = REPO_DIR / 'raw_data' / f'{REGION}_sentinel1_corrected'
DIY_DIR = REPO_DIR / 'raw_data' / f'{REGION}_sentinel1_diy_corrected'
OUT_DIR = REPO_DIR / 'input_data' / f's1_patches_{REGION}'
USE_CORRECTED_GEOTIFFS = False
DST_RESOLUTION_M = 10.0
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Calibration helpers for raw SAFE products

Sentinel-1 calibration XML provides a sparse sigma-nought lookup grid. The grid is interpolated to measurement resolution, then the calibrated raster is warped using the product GCPs.

In [ ]:
def find_pol_files(safe_dir, pol):
    measurement = next(Path(safe_dir).glob(f'measurement/*-{pol}-*.tiff'))
    calibration = next(Path(safe_dir).glob(f'annotation/calibration/calibration-*-{pol}-*.xml'))
    return measurement, calibration

def calibration_lut(xml_path, shape):
    root = ET.parse(xml_path).getroot()
    vectors = root.find('calibrationVectorList').findall('calibrationVector')
    lines, pixels, values = [], None, []
    for vector in vectors:
        lines.append(int(vector.find('line').text))
        current_pixels = np.array([int(x) for x in vector.find('pixel').text.split()], dtype=float)
        pixels = current_pixels if pixels is None else pixels
        values.append([float(x) for x in vector.find('sigmaNought').text.split()])
    interpolator = RectBivariateSpline(np.asarray(lines, float), pixels, np.asarray(values, float), kx=1, ky=1)
    return interpolator(np.arange(shape[0]), np.arange(shape[1])).astype(np.float32)

def warp_grid(measurement_path, dst_crs, resolution):
    with rasterio.open(measurement_path) as src:
        gcps, gcp_crs = src.gcps
        transform, width, height = calculate_default_transform(gcp_crs, dst_crs, src.width, src.height, gcps=gcps, resolution=(resolution, resolution))
    return transform, width, height, gcp_crs

def calibrate_and_warp(measurement_path, calibration_path, dst_crs, output_path, grid):
    transform, width, height, gcp_crs = grid
    with rasterio.open(measurement_path) as src:
        dn = src.read(1).astype(np.float32)
        gcps, _ = src.gcps
    lut = calibration_lut(calibration_path, dn.shape)
    with np.errstate(divide='ignore', invalid='ignore'):
        sigma0 = np.where(lut > 0, (dn ** 2) / (lut ** 2), np.nan).astype(np.float32)
    profile = {'driver': 'GTiff', 'dtype': 'float32', 'count': 1, 'width': width, 'height': height, 'crs': dst_crs, 'transform': transform, 'nodata': np.nan}
    with rasterio.open(output_path, 'w', **profile) as dst:
        reproject(sigma0, rasterio.band(dst, 1), gcps=gcps, src_crs=gcp_crs, dst_transform=transform, dst_crs=dst_crs, resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    return output_path

## Build analysis-ready Sentinel-1 products

For each product, VV and VH are warped onto the same grid and stacked as bands 1 and 2. If corrected GeoTIFFs already exist, set `USE_CORRECTED_GEOTIFFS = True`.

In [ ]:
def build_products():
    if USE_CORRECTED_GEOTIFFS:
        return sorted(CORRECTED_DIR.glob('*.tif'))
    safe_dirs = sorted(SAFE_ROOT.glob('*/**/*.SAFE'))
    lidar_ref = next(LIDAR_DIR.glob('lidar_patch_*.tif'))
    with rasterio.open(lidar_ref) as ref:
        dst_crs = ref.crs
    paths = []
    DIY_DIR.mkdir(parents=True, exist_ok=True)
    for index, safe_dir in enumerate(safe_dirs):
        vv, vv_xml = find_pol_files(safe_dir, 'vv')
        vh, vh_xml = find_pol_files(safe_dir, 'vh')
        grid = warp_grid(vv, dst_crs, DST_RESOLUTION_M)
        vv_out = DIY_DIR / f'_vv_{index}.tif'
        vh_out = DIY_DIR / f'_vh_{index}.tif'
        calibrate_and_warp(vv, vv_xml, dst_crs, vv_out, grid)
        calibrate_and_warp(vh, vh_xml, dst_crs, vh_out, grid)
        final = DIY_DIR / f't{index}.tif'
        with rasterio.open(vv_out) as a, rasterio.open(vh_out) as b:
            meta = a.meta.copy(); meta.update(count=2)
            with rasterio.open(final, 'w', **meta) as dst:
                dst.write(a.read(1), 1); dst.write(b.read(1), 2)
        vv_out.unlink(); vh_out.unlink(); paths.append(final)
    return paths

products = build_products()
print('Analysis-ready products:', [str(p) for p in products])

## Match products to LiDAR patches

The Sentinel-1 window is selected from each LiDAR patch's georeferenced bounds. The output keeps the temporal naming convention used by Tessa's loader. For the later model notebook, VV and VH are converted to dB and adapted to the four-channel conditioning interface.

In [ ]:
def collocate(products, lidar_dir, output_dir, resolution=10.0):
    for lidar_path in sorted(Path(lidar_dir).glob('lidar_patch_*.tif')):
        patch_id = lidar_path.stem.split('_')[-1]
        with rasterio.open(lidar_path) as lidar:
            bounds, lidar_crs = lidar.bounds, lidar.crs
        patch_dir = Path(output_dir) / f's1_patch_{patch_id}'
        patch_dir.mkdir(parents=True, exist_ok=True)
        attrs = []
        for i, product_path in enumerate(products):
            with rasterio.open(product_path) as src:
                sb = transform_bounds(lidar_crs, src.crs, *bounds, densify_pts=21)
                window = from_bounds(*sb, transform=src.transform).round_offsets().round_lengths()
                patch = src.read(window=window)
                profile = src.profile.copy(); profile.update(height=patch.shape[1], width=patch.shape[2], transform=rasterio.windows.transform(window, src.transform), count=patch.shape[0], dtype='float32', nodata=np.nan)
            if patch.shape[1] == 0 or patch.shape[2] == 0 or not np.isfinite(patch).any():
                continue
            with rasterio.open(patch_dir / f't{i}.tif', 'w', **profile) as dst:
                dst.write(patch.astype(np.float32))
            attrs.append({'source': str(product_path), 'polarizations': ['VV', 'VH'], 'acquisition_date': None, 'cloud_cover': 0.0})
        with (patch_dir / 'attrs.json').open('w') as handle:
            json.dump(attrs, handle, indent=2)

collocate(products, LIDAR_DIR, OUT_DIR, DST_RESOLUTION_M)
print('Collocation complete:', OUT_DIR)